# stock-bqml POC Pipeline\nBigQuery ML で株価の翌日リターン予測を体験する。\n\n- Silver: テクニカル指標生成\n- Gold: XGBoost 学習・予測・解釈

In [ ]:
# G CP 設定\nPROJECT_ID = 'your-gcp-project'\nREGION = 'US'\n\nfrom google.colab import auth\nauth.authenticate_user()\n!gcloud config set project {PROJECT_ID}

## 1. Silver 特徴量生成

In [ ]:
# features_daily.sql を実行\n!bq query --use_legacy_sql=false \\n  --replace \\n  --destination_table={PROJECT_ID}:stock_silver.features_daily \\n  < silver/features_daily.sql

## 2. Gold モデル学習

In [ ]:
!bq query --use_legacy_sql=false < gold/train_xgb.sql

## 3. 予測

In [ ]:
!bq query --use_legacy_sql=false < gold/predict.sql

## 4. 解釈（特徴量重要度）

In [ ]:
import pandas as pd\ndf = pd.read_gbq('SELECT * FROM ML.GLOBAL_EXPLAIN(MODEL `stock_gold.xgb_predictor`)', project_id=PROJECT_ID)\ndf.head(10)

## Appendix: 結果可視化

In [ ]:
pred = pd.read_gbq('SELECT * FROM `stock_gold.predictions` ORDER BY date DESC LIMIT 100', project_id=PROJECT_ID)\npred.plot(x='date', y='predicted_next_day_return', figsize=(12,4))